# 1. Weather Stream Producer

This notebook simulates weather sensor events and publishes them to Kafka. It reads transformed weather records in chronological order, emits one simulated day per second, attaches `weather_ts` at send time, and stores per-site pointers so the stream can resume cleanly.

In [3]:
# Weather stream producer - read weather rows and send one simulated day per second

from time import sleep
import json, time, datetime as dt
from pathlib import Path
import pandas as pd
import numpy as np
from json import dumps
from kafka3 import KafkaProducer

#CONFIGURATIONS
WEATHER_CSV = "../A2A/weather_transf.csv"
HOST_IP = "kafka"
TOPIC = "weather_stream"

BATCH_INTERVAL_SECONDS = 5
DAYS_PER_BATCH = 5
ROWS_PER_DAY = 24
SITE_IDS = list(range(16))  # define all of the sites present

POINTER_DIR = Path("site_seq_pointers"); POINTER_DIR.mkdir(exist_ok=True) #Pointers save at which "point" we are so it can be resumed later

def _pointer_path(sid): return POINTER_DIR / f"pointer_{sid}.json"

def load_pointer(sid):
    p = _pointer_path(sid)
    if not p.exists(): return 0
    try: return int(json.loads(p.read_text()).get("day_idx", 0))
    except: return 0
    
def save_pointer(sid, idx): _pointer_path(sid).write_text(json.dumps({"day_idx": int(idx)}))

def connect_kafka_producer():
    return KafkaProducer(bootstrap_servers=[f"{HOST_IP}:9092"], api_version=(0,10))

def publish(producer, topic, key, value):
    return producer.send(topic, key=key.encode(), value=value.encode())

#  Load data and partition
df = pd.read_csv(WEATHER_CSV)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).sort_values(["site_id","timestamp"]).reset_index(drop=True)
df["date"] = df["timestamp"].dt.date

site_frames = {sid: g.copy() for sid, g in df.groupby("site_id", sort=True)}
site_days   = {sid: sorted(g["date"].unique()) for sid, g in df.groupby("site_id", sort=True)}


def pad_day_24(df_day, site_id, day) :
# Pad a single day to 24 hourly rows (leave missing values as None), if same hour has multiple entries, keep average

    start = pd.Timestamp(day)
    hours = pd.date_range(start, start + pd.Timedelta(hours=23), freq="H")

    if df_day.empty:
        g = pd.DataFrame(index=hours)
    else:
        g = df_day.copy()
        g["timestamp"] = pd.to_datetime(g["timestamp"])
        g["ts_hour"] = g["timestamp"].dt.floor("H")

        num_cols = g.select_dtypes(include=[np.number]).columns.tolist()
        agg_map = {c: "mean" for c in num_cols}
        for c in g.columns:
            if c not in agg_map and c != "ts_hour":
                agg_map[c] = "first"

        g = (g.groupby("ts_hour", as_index=True).agg(agg_map)
               .reindex(hours))

    if "timestamp" in g.columns:
        g = g.drop(columns=["timestamp"])
    g.index.name = "timestamp"
    g = g.reset_index()
    g["site_id"] = site_id
    g["date"] = pd.Timestamp(day).date()
    return g

def get_next_window_padded(sid, start_idx, k) :
    days = site_days[sid]
    sel = [days[(start_idx + i) % len(days)] for i in range(k)]
    gsite = site_frames[sid]
    parts = []
    for d in sel:
        day_slice = gsite[gsite["date"] == d]
        parts.append(pad_day_24(day_slice, sid, d))
    out = pd.concat(parts, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    next_idx = (start_idx + k) % len(days)
    return out, next_idx, sel

def to_jsonable(row):
    def fix(v):
        if v is None: return None
        try:
            if pd.isna(v): return None
        except Exception:
            pass
        return v.item() if hasattr(v, "item") else v
    return {k: fix(v) for k, v in row.items()}

producer = connect_kafka_producer()
assert producer is not None, "Kafka not reachable"

#Reading each site sequentially starting from site 0 (so we don't have issues with time order)
print(f"Sequential streaming {len(SITE_IDS)} sites -> topic '{TOPIC}'")

for sid in SITE_IDS:

    day_idx = load_pointer(sid) #loads current point in processing 
    days = site_days[sid]
    print(f"[site {sid}] start day_idx={day_idx}, total_days={len(days)}")

    while True:
        batch_df, next_idx, sel_days = get_next_window_padded(sid, day_idx, DAYS_PER_BATCH)
        sent = 0
        sent_ts = []
        
        # Emit one simulated day per second. This makes weather_ts match arrival time
        # instead of publishing the full five-day batch instantly and sleeping afterwards.
        for day_offset in range(DAYS_PER_BATCH):
            start_pos = day_offset * ROWS_PER_DAY
            end_pos = start_pos + ROWS_PER_DAY
            day_rows = batch_df.iloc[start_pos:end_pos]
            send_started_at = time.time()
            send_ts = int(send_started_at)
            sent_ts.append(send_ts)
            futures = []
            
            for _, r in day_rows.iterrows():
                payload = r.to_dict()
                payload["timestamp"] = pd.Timestamp(payload["timestamp"]).isoformat()
                payload.pop("date", None)
                payload = to_jsonable(payload)
                payload["weather_ts"] = send_ts
                futures.append(publish(producer, TOPIC, f"site-{sid}", dumps(payload, separators=(",",":"))))
                sent += 1
            
            producer.flush()
            for fut in futures:
                fut.get(timeout=10)
            
            elapsed = time.time() - send_started_at
            if day_offset < DAYS_PER_BATCH - 1:
                sleep(max(0, 1.0 - elapsed))

        save_pointer(sid, next_idx) #saves current state
        day_idx = next_idx
        
        #VISUAL PROGRESSION, useful for debugging
        print(f"[{dt.datetime.now().strftime('%X')}] [site {sid}] Sent {sent} rows "
              f"(days {sel_days[0]}..{sel_days[-1]}), weather_ts {min(sent_ts)}..{max(sent_ts)}. "
              f"next day_idx={day_idx}")

        # Stop when we have looped through all days once (finished this site)
        if day_idx == 0: 
            print(f"[site {sid}] completed all days, moving to next site") #finished x site in sequence
            break

print("All sites completed.")

Sequential streaming 16 sites -> topic 'weather_stream'
[site 0] start day_idx=275, total_days=365
[13:09:05] [site 0] Sent 120 rows (days 2022-10-03..2022-10-07), weather_ts 1782306541..1782306545. next day_idx=280
[13:09:09] [site 0] Sent 120 rows (days 2022-10-08..2022-10-12), weather_ts 1782306545..1782306549. next day_idx=285
[13:09:13] [site 0] Sent 120 rows (days 2022-10-13..2022-10-17), weather_ts 1782306549..1782306553. next day_idx=290
[13:09:15] [site 0] Sent 120 rows (days 2022-10-18..2022-10-22), weather_ts 1782306552..1782306555. next day_idx=295
[13:09:19] [site 0] Sent 120 rows (days 2022-10-23..2022-10-27), weather_ts 1782306555..1782306559. next day_idx=300
[13:09:23] [site 0] Sent 120 rows (days 2022-10-28..2022-11-01), weather_ts 1782306559..1782306563. next day_idx=305
[13:09:27] [site 0] Sent 120 rows (days 2022-11-02..2022-11-06), weather_ts 1782306563..1782306567. next day_idx=310
[13:09:32] [site 0] Sent 120 rows (days 2022-11-07..2022-11-11), weather_ts 178230

KeyboardInterrupt: 

**Producer behaviour notes**

- Sites are streamed sequentially, starting from site 0.
- Each emitted window contains 5 simulated days, with one day spread over each second of the 5-second interval.
- Missing hourly periods are padded so downstream processing receives a consistent 24-row daily shape.